<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin-right: 20px; height: 55px" height="55px">

# 4. Guardrails, Observability & Evaluation (Solved Reference)

Fully worked solutions to all three exercises. Use this to check your work or to
catch up if you fell behind during the live lab. See `solutions folder` for
prose explanations alongside this code.

---

## Solution Guide

1. [Shared setup: TF-IDF-weighted retrieval and composite guardrail](#exercise0)
2. [Exercise 1 — Calibrating the Composite Guardrail](#exercise1)
3. [Exercise 2 — CI Regression Gate with Multi-Run Averaging](#exercise2)
4. [Exercise 3 — Budget Violations and p95 Aggregation](#exercise3)

---

In [91]:
!pip install ragas langsmith langchain langchain_openai openai langchain-community==0.2.16 langchain-huggingface faiss-cpu sentence-transformers datasets langchain-google-vertexai
!apt-get update -qq
!apt-get install -y -qq zstd

!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
import os
import getpass
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)

# Set your OpenAI API Key here if you have one. This is optional.
openai_key = os.getenv("OPENAI_API_KEY") or getpass.getpass("Enter your OpenAI API Key (leave empty to skip): ")
if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key

print("API key input prompts added.")


# Set your Langsmith API Key here if you have one. This is optional.
langsmith_key = os.getenv("LANGSMITH_API_KEY") or getpass.getpass("Enter your Langsmith API Key (leave empty to skip): ")
if langsmith_key:
    os.environ["LANGSMITH_API_KEY"] = langsmith_key

print("API key input prompts added.")

Enter your OpenAI API Key (leave empty to skip): ··········
API key input prompts added.
Enter your Langsmith API Key (leave empty to skip): ··········
API key input prompts added.


In [2]:
import json
import os

_candidate_paths = ["../data/golden_dataset_module4.json", "data/golden_dataset_module4.json"]
golden = None
for _path in _candidate_paths:
    if os.path.exists(_path):
        golden = json.load(open(_path))
        break

if golden is None:
    golden = json.loads('''{
  "knowledge_base": [
    {
      "doc_id": "kb_001",
      "text": "To reset your Acme account password, go to Settings > Security > Reset Password and follow the emailed link. The link expires after 30 minutes."
    },
    {
      "doc_id": "kb_002",
      "text": "VPN error 691 indicates an authentication failure. Confirm your username and password are correct and that your account is not locked before contacting IT."
    },
    {
      "doc_id": "kb_003",
      "text": "Employees may expense client dinners up to $75 per person with an itemized receipt and the client's name and company noted on the expense report."
    },
    {
      "doc_id": "kb_004",
      "text": "2019 PTO Policy: Full-time employees accrue 15 days of paid time off per year, credited monthly."
    },
    {
      "doc_id": "kb_005",
      "text": "2024 PTO Policy Addendum (supersedes the 2019 policy): Full-time employees now receive unlimited PTO, subject to manager approval and a minimum of 10 days taken per year."
    },
    {
      "doc_id": "kb_006",
      "text": "The remote work stipend of $50/month for home internet does not apply to employees on the Contractor or Intern employment tracks."
    },
    {
      "doc_id": "kb_007",
      "text": "Benefits enrollment opens each year in the first two weeks of November. Changes take effect on January 1st of the following year."
    },
    {
      "doc_id": "kb_008",
      "text": "If your office printer shows an offline error, check that it is connected to the 'Acme-Print' network and restart the print spooler service."
    },
    {
      "doc_id": "kb_009",
      "text": "Guests can connect to the 'Acme-Guest' wifi network using the daily password posted at the front desk; it does not require a company account."
    },
    {
      "doc_id": "kb_010",
      "text": "Expense report code EXP-114 is used for software subscription reimbursements under $50/month that do not require manager pre-approval."
    },
    {
      "doc_id": "kb_011",
      "text": "Security incidents, including suspected phishing emails or lost devices, must be reported to security@acme.example within 1 hour of discovery."
    },
    {
      "doc_id": "kb_012",
      "text": "Customer data is retained for 7 years after account closure to meet financial audit requirements, then permanently deleted."
    },
    {
      "doc_id": "kb_013",
      "text": "Parental leave provides 16 weeks of fully paid leave for the primary caregiver and 6 weeks for the secondary caregiver, available to all full-time employees."
    },
    {
      "doc_id": "kb_014",
      "text": "Employee referral bonuses are $2,000 for standard roles and $4,000 for senior engineering roles, paid out after the new hire completes 90 days."
    },
    {
      "doc_id": "kb_015",
      "text": "Company laptops are eligible for replacement every 3 years, or sooner if hardware fails and cannot be repaired by IT within 5 business days."
    },
    {
      "doc_id": "kb_016",
      "text": "Home office equipment reimbursement covers up to $300 one time for a monitor, chair, or keyboard, submitted through expense code EXP-220."
    },
    {
      "doc_id": "kb_017",
      "text": "Slow application performance is often caused by too many browser tabs or an outdated client version; check for updates under Help > About."
    },
    {
      "doc_id": "kb_018",
      "text": "The office building requires badge access after 7pm on weekdays and at all times on weekends; lost badges should be reported to facilities immediately."
    }
  ],
  "golden_dataset": [
    {
      "id": "q01",
      "question": "How do I reset my Acme account password?",
      "ground_truth": "Go to Settings > Security > Reset Password and use the emailed link within 30 minutes.",
      "is_answerable": true,
      "relevant_doc_ids": [
        "kb_001"
      ]
    },
    {
      "id": "q02",
      "question": "I'm getting VPN error 691, what does that mean?",
      "ground_truth": "It's an authentication failure; verify your username/password and that your account isn't locked.",
      "is_answerable": true,
      "relevant_doc_ids": [
        "kb_002"
      ]
    },
    {
      "id": "q03",
      "question": "Can I expense a client dinner and how much is covered?",
      "ground_truth": "Yes, up to $75 per person with an itemized receipt and the client's name and company noted.",
      "is_answerable": true,
      "relevant_doc_ids": [
        "kb_003"
      ]
    },
    {
      "id": "q04",
      "question": "How many PTO days do full-time employees get per year?",
      "ground_truth": "Unlimited PTO as of the 2024 policy, subject to manager approval and a 10-day minimum taken per year (this supersedes the old 2019 15-day accrual policy).",
      "is_answerable": true,
      "relevant_doc_ids": [
        "kb_005"
      ]
    },
    {
      "id": "q05",
      "question": "Does the remote work stipend apply to interns?",
      "ground_truth": "No, the $50/month stipend does not apply to Contractor or Intern employment tracks.",
      "is_answerable": true,
      "relevant_doc_ids": [
        "kb_006"
      ]
    },
    {
      "id": "q06",
      "question": "When does benefits enrollment open?",
      "ground_truth": "The first two weeks of November each year, with changes effective January 1st.",
      "is_answerable": true,
      "relevant_doc_ids": [
        "kb_007"
      ]
    },
    {
      "id": "q07",
      "question": "My office printer says it's offline, what should I check?",
      "ground_truth": "Check it's connected to the Acme-Print network and restart the print spooler service.",
      "is_answerable": true,
      "relevant_doc_ids": [
        "kb_008"
      ]
    },
    {
      "id": "q08",
      "question": "How do guests connect to wifi in the office?",
      "ground_truth": "Guests use the Acme-Guest network with the daily password posted at the front desk; no company account needed.",
      "is_answerable": true,
      "relevant_doc_ids": [
        "kb_009"
      ]
    },
    {
      "id": "q09",
      "question": "What expense code do I use for a software subscription under $50 a month?",
      "ground_truth": "Use expense code EXP-114, which doesn't require manager pre-approval for subscriptions under $50/month.",
      "is_answerable": true,
      "relevant_doc_ids": [
        "kb_010"
      ]
    },
    {
      "id": "q10",
      "question": "How quickly do I need to report a lost company device?",
      "ground_truth": "Within 1 hour of discovery, to security@acme.example.",
      "is_answerable": true,
      "relevant_doc_ids": [
        "kb_011"
      ]
    },
    {
      "id": "q11",
      "question": "How long is customer data retained after account closure?",
      "ground_truth": "7 years after account closure, then it is permanently deleted.",
      "is_answerable": true,
      "relevant_doc_ids": [
        "kb_012"
      ]
    },
    {
      "id": "q12",
      "question": "How much parental leave do primary caregivers get?",
      "ground_truth": "16 weeks fully paid for the primary caregiver, 6 weeks for the secondary caregiver.",
      "is_answerable": true,
      "relevant_doc_ids": [
        "kb_013"
      ]
    },
    {
      "id": "q13",
      "question": "What's the referral bonus for a senior engineering hire?",
      "ground_truth": "$4,000, paid out after the new hire completes 90 days.",
      "is_answerable": true,
      "relevant_doc_ids": [
        "kb_014"
      ]
    },
    {
      "id": "q14",
      "question": "How often can I get my company laptop replaced?",
      "ground_truth": "Every 3 years, or sooner if it fails and IT can't repair it within 5 business days.",
      "is_answerable": true,
      "relevant_doc_ids": [
        "kb_015"
      ]
    },
    {
      "id": "q15",
      "question": "What's the CEO's personal cell phone number?",
      "ground_truth": "This information is not available in the knowledge base; the assistant should decline to answer.",
      "is_answerable": false,
      "relevant_doc_ids": []
    },
    {
      "id": "q16",
      "question": "What will Acme's stock price be next quarter?",
      "ground_truth": "This information is not available in the knowledge base; the assistant should decline to answer.",
      "is_answerable": false,
      "relevant_doc_ids": []
    },
    {
      "id": "q17",
      "question": "Can I get a company car as part of my benefits package?",
      "ground_truth": "This information is not available in the knowledge base; the assistant should decline to answer.",
      "is_answerable": false,
      "relevant_doc_ids": []
    }
  ]
}''')

KNOWLEDGE_BASE = golden["knowledge_base"]
GOLDEN_DATASET = golden["golden_dataset"]
print(f"Loaded {len(KNOWLEDGE_BASE)} knowledge-base docs and {len(GOLDEN_DATASET)} eval queries.")


Loaded 18 knowledge-base docs and 17 eval queries.


<h2 id="exercise0"> Shared setup: TF-IDF-weighted retrieval and composite guardrail </h2>

In [3]:
import re, math, random, time, json as json_module
from collections import Counter

STOPWORDS = {"the","a","an","is","are","to","of","and","in","on","for","my","i",
             "how","what","do","does","can","this","that","it","be","will","you","your","need"}

def lemma(word: str) -> str:
    for suffix in ("ing", "ies", "ed", "es", "s"):
        if word.endswith(suffix) and len(word) > len(suffix) + 2:
            return word[: -len(suffix)]
    return word

def tokenize(text: str) -> list:
    words = re.sub(r"[^a-z0-9\s]", " ", text.lower()).split()
    return [lemma(w) for w in words if w not in STOPWORDS]

N_DOCS = len(KNOWLEDGE_BASE)
df_counter = Counter()
for doc in KNOWLEDGE_BASE:
    df_counter.update(set(tokenize(doc["text"])))

def idf(word: str) -> float:
    df = df_counter.get(word, 0)
    return math.log((N_DOCS + 1) / (df + 1)) + 1

def retrieval_score(query: str, doc_text: str) -> float:
    q_tokens = tokenize(query)
    d_tokens = set(tokenize(doc_text))
    if not q_tokens:
        return 0.0
    matched_weight = sum(idf(w) for w in q_tokens if w in d_tokens)
    total_weight = sum(idf(w) for w in q_tokens)
    return matched_weight / total_weight if total_weight else 0.0

def retrieve(query: str, top_k: int = 5) -> list:
    scored = [(d["doc_id"], d["text"], retrieval_score(query, d["text"])) for d in KNOWLEDGE_BASE]
    scored.sort(key=lambda x: -x[2])
    return scored[:top_k]

def composite_guardrail(query: str, top_k: int = 5, score_floor: float = 0.35,
                         min_margin: float = 0.10, corroboration_floor: float = 0.25) -> dict:
    results = retrieve(query, top_k=top_k)
    top_score = results[0][2] if results else 0.0
    second_score = results[1][2] if len(results) > 1 else 0.0
    margin = top_score - second_score
    corroboration = sum(1 for _, _, s in results if s >= corroboration_floor)
    should_answer = (top_score >= score_floor) and (margin >= min_margin or corroboration <= 1)
    reason = []
    if top_score < score_floor:
        reason.append(f"top_score {top_score:.2f} below floor {score_floor}")
    if margin < min_margin and corroboration > 1:
        reason.append(f"ambiguous: margin {margin:.2f} with {corroboration} corroborating candidates")
    return {"should_answer": should_answer, "top_score": top_score, "margin": margin,
             "corroboration": corroboration,
             "reason": "; ".join(reason) if reason else "confident single best match"}


<h2 id="exercise1"> Exercise 1 — Calibrating the Composite Guardrail </h2>

In [4]:
def calibrate_guardrail(dataset: list, **guardrail_kwargs) -> dict:
    tp = fp = tn = fn = 0
    for item in dataset:
        result = composite_guardrail(item["question"], **guardrail_kwargs)
        predicted, actual = result["should_answer"], item["is_answerable"]
        if predicted and actual:
            tp += 1
        elif predicted and not actual:
            fp += 1
            print(f"  FALSE POSITIVE: {item['id']} -- {item['question']}")
        elif not predicted and actual:
            fn += 1
            print(f"  FALSE NEGATIVE: {item['id']} -- {item['question']} "
                  f"(top_score={result['top_score']:.2f}, margin={result['margin']:.2f}, "
                  f"corroboration={result['corroboration']})")
        else:
            tn += 1

    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    return {"tp": tp, "fp": fp, "tn": tn, "fn": fn, "precision": precision, "recall": recall}


In [5]:
print("=== Default threshold: score_floor=0.35 ===")
r1 = calibrate_guardrail(GOLDEN_DATASET, score_floor=0.35)
print(f"TP={r1['tp']} FP={r1['fp']} TN={r1['tn']} FN={r1['fn']}  "
      f"Precision={r1['precision']:.0%}  Recall={r1['recall']:.0%}")

print("\n=== Lowered threshold: score_floor=0.30 ===")
r2 = calibrate_guardrail(GOLDEN_DATASET, score_floor=0.30)
print(f"TP={r2['tp']} FP={r2['fp']} TN={r2['tn']} FN={r2['fn']}  "
      f"Precision={r2['precision']:.0%}  Recall={r2['recall']:.0%}")

assert r2["fp"] == 0, "Lowering the floor should not introduce false positives"
assert r2["recall"] > r1["recall"], "Lowering the floor should improve recall"
print("\nConfirmed: recall improved with zero false-positive cost.")


=== Default threshold: score_floor=0.35 ===
  FALSE NEGATIVE: q04 -- How many PTO days do full-time employees get per year? (top_score=0.70, margin=0.00, corroboration=3)
  FALSE NEGATIVE: q10 -- How quickly do I need to report a lost company device? (top_score=0.33, margin=0.00, corroboration=3)
  FALSE NEGATIVE: q14 -- How often can I get my company laptop replaced? (top_score=0.34, margin=0.15, corroboration=1)
TP=11 FP=0 TN=3 FN=3  Precision=100%  Recall=79%

=== Lowered threshold: score_floor=0.30 ===
  FALSE NEGATIVE: q04 -- How many PTO days do full-time employees get per year? (top_score=0.70, margin=0.00, corroboration=3)
  FALSE NEGATIVE: q10 -- How quickly do I need to report a lost company device? (top_score=0.33, margin=0.00, corroboration=3)
TP=12 FP=0 TN=3 FN=2  Precision=100%  Recall=86%

Confirmed: recall improved with zero false-positive cost.


<h2 id="exercise2"> Exercise 2 — CI Regression Gate with Multi-Run Averaging </h2>

In [6]:
def simulated_generate(query: str, context_texts: list, hallucinate: bool = False) -> str:
    grounded_answer = context_texts[0] if context_texts else "I don't have information on this."
    if not hallucinate:
        return grounded_answer
    fabricated = "This also qualifies for an additional 20 percent bonus not mentioned anywhere in company policy."
    return grounded_answer + " " + fabricated


def split_sentences(text: str) -> list:
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def faithfulness(answer: str, context_texts: list, support_threshold: float = 0.4) -> float:
    context_tokens = set()
    for c in context_texts:
        context_tokens |= set(tokenize(c))
    sentences = split_sentences(answer)
    if not sentences:
        return 1.0
    supported = sum(
        1 for s in sentences
        if not tokenize(s) or len(set(tokenize(s)) & context_tokens) / len(set(tokenize(s))) >= support_threshold
    )
    return supported / len(sentences)

def context_precision(retrieved_doc_ids: list, relevant_doc_ids: list) -> float:
    if not retrieved_doc_ids:
        return 0.0
    return sum(1 for d in retrieved_doc_ids if d in relevant_doc_ids) / len(retrieved_doc_ids)

def answer_relevancy(answer: str, question: str) -> float:
    return retrieval_score(question, answer)

def simulated_llm_judge_noise(true_score: float, rng: random.Random) -> float:
    return max(0.0, min(1.0, true_score + rng.uniform(-0.05, 0.05)))

def evaluate_dataset(hallucinate: bool = False, top_k: int = 3) -> dict:
    precisions, faiths, relevancies = [], [], []
    for item in GOLDEN_DATASET:
        if not item["is_answerable"]:
            continue
        retrieved = retrieve(item["question"], top_k=top_k)
        retrieved_ids = [d[0] for d in retrieved]
        context_texts = [d[1] for d in retrieved]
        answer = simulated_generate(item["question"], context_texts, hallucinate=hallucinate)
        precisions.append(context_precision(retrieved_ids, item["relevant_doc_ids"]))
        faiths.append(faithfulness(answer, context_texts))
        relevancies.append(answer_relevancy(answer, item["question"]))
    n = len(precisions)
    return {"avg_context_precision": sum(precisions)/n, "avg_faithfulness": sum(faiths)/n,
            "avg_answer_relevancy": sum(relevancies)/n}


def stable_evaluate_faithfulness(hallucinate: bool, n_runs: int = 5) -> float:
    rng = random.Random(123)
    scores = []
    for _ in range(n_runs):
        result = evaluate_dataset(hallucinate=hallucinate)
        noisy = simulated_llm_judge_noise(result["avg_faithfulness"], rng)
        scores.append(noisy)
    return sum(scores) / len(scores)

def ci_gate(hallucinate: bool, tolerance: float = 0.05):
    current = stable_evaluate_faithfulness(hallucinate=hallucinate, n_runs=10)
    passed = current >= BASELINE_FAITHFULNESS - tolerance
    return passed, current


In [7]:
BASELINE_FAITHFULNESS = stable_evaluate_faithfulness(hallucinate=False, n_runs=10)
print(f"Baseline (faithful) avg faithfulness over 10 runs: {BASELINE_FAITHFULNESS:.3f}")

passed, score = ci_gate(hallucinate=False)
print(f"Gate on unchanged pipeline:            passed={passed}, score={score:.3f}")
assert passed, "Gate should pass on the unchanged (faithful) pipeline"

passed, score = ci_gate(hallucinate=True)
print(f"Gate on regressed (hallucinating) pipeline: passed={passed}, score={score:.3f}")
assert not passed, "Gate should fail on the regressed (hallucinating) pipeline"

print("\nConfirmed: gate passes on baseline, fails on regression.")


Baseline (faithful) avg faithfulness over 10 runs: 0.977
Gate on unchanged pipeline:            passed=True, score=0.977
Gate on regressed (hallucinating) pipeline: passed=False, score=0.520

Confirmed: gate passes on baseline, fails on regression.


<h2 id="exercise3"> Exercise 3 — Budget Violations and p95 Aggregation </h2>

In [8]:
class Tracer:
    def __init__(self):
        self.spans = []
    def span(self, name, **metadata):
        return _Span(self, name, metadata)
    def log(self, record):
        self.spans.append(record)
    def to_jsonl(self):
        return "\n".join(json_module.dumps(r) for r in self.spans)

class _Span:
    def __init__(self, tracer, name, metadata):
        self.tracer, self.name, self.metadata = tracer, name, metadata
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed_ms = (time.perf_counter() - self.start) * 1000
        self.tracer.log({"span": self.name, "latency_ms": round(elapsed_ms, 3), **self.metadata})


SLOW_QUERY_IDS = {"q08"}

def traced_pipeline_run_with_incident(query_id: str, query: str, tracer: Tracer,
                                       guardrail_score_floor: float = 0.30) -> str:
    with tracer.span("retrieval", query_id=query_id) as s:
        retrieved = retrieve(query, top_k=5)
        s.metadata["doc_count"] = len(retrieved)
        s.metadata["top_score"] = round(retrieved[0][2], 3) if retrieved else 0.0

    with tracer.span("guardrail", query_id=query_id) as s:
        guardrail_result = composite_guardrail(query, score_floor=guardrail_score_floor)
        s.metadata["should_answer"] = guardrail_result["should_answer"]

    if not guardrail_result["should_answer"]:
        with tracer.span("fallback", query_id=query_id) as s:
            answer = "I don't know based on the available information."
            s.metadata["generation_skipped"] = True
        return answer

    with tracer.span("generation", query_id=query_id) as s:
        context_texts = [d[1] for d in retrieved[:3]]
        answer = simulated_generate(query, context_texts, hallucinate=False)
        time.sleep(0.15 if query_id in SLOW_QUERY_IDS else 0.05)
        s.metadata["token_estimate"] = len(answer.split())

    return answer


def check_budget_violations(tracer: Tracer, budget_ms: dict) -> list:
    return [r for r in tracer.spans if r["span"] in budget_ms and r["latency_ms"] > budget_ms[r["span"]]]

def p95(values: list):
    if not values:
        return None
    sorted_vals = sorted(values)
    idx = int(0.95 * (len(sorted_vals) - 1))
    return sorted_vals[idx]

def aggregate_latency_by_span(tracer: Tracer) -> dict:
    by_span = {}
    for r in tracer.spans:
        by_span.setdefault(r["span"], []).append(r["latency_ms"])
    return {span: {"p95_ms": p95(v), "count": len(v)} for span, v in by_span.items()}


In [9]:
tracer = Tracer()
for item in GOLDEN_DATASET:
    traced_pipeline_run_with_incident(item["id"], item["question"], tracer, guardrail_score_floor=0.30)

BUDGET_MS = {"retrieval": 50, "guardrail": 10, "generation": 100, "fallback": 20}
violations = check_budget_violations(tracer, BUDGET_MS)

print(f"Total spans logged: {len(tracer.spans)}")
print(f"Budget violations found: {len(violations)}")
for v in violations:
    print(f"  query_id={v['query_id']} span={v['span']} latency={v['latency_ms']:.1f}ms (budget {BUDGET_MS[v['span']]}ms)")

assert len(violations) == 1 and violations[0]["query_id"] == "q08", \
    "Should find exactly one violation, on q08"

print()
agg = aggregate_latency_by_span(tracer)
for span, stats in agg.items():
    print(f"{span:<12} p95={stats['p95_ms']:.2f}ms  n={stats['count']}")

print("\nConfirmed: exactly one budget violation found, correctly attributed to q08.")


Total spans logged: 51
Budget violations found: 1
  query_id=q08 span=generation latency=150.2ms (budget 100ms)

retrieval    p95=0.71ms  n=17
guardrail    p95=0.59ms  n=17
generation   p95=50.18ms  n=12
fallback     p95=0.00ms  n=5

Confirmed: exactly one budget violation found, correctly attributed to q08.


---

# Extension — Cloud/API Models Approach


> **The code cells below require either a paid API key (Cloud/API track) or a local model download**

**Guardrails, Observability & Evaluation**

This section shows the real *cloud/API* tooling equivalents for evaluation and
observability — this is also the most common real-world setup, since Ragas and most
tracing backends are typically paired with a hosted LLM judge in practice.

### What Changes, Component by Component

| Course concept | Course's stand-in | Cloud/API equivalent |
|---|---|---|
| Hallucination guardrail | Retrieval-signal composite guardrail | Unchanged — retrieval-signal-based, not model-based, in both approaches |
| RAG Triad evaluation | Lexical-overlap metric functions | `ragas` with `langchain_openai.ChatOpenAI` as judge (the standard, most-documented Ragas setup) |
| LLM-judge noise / averaging | `simulated_llm_judge_noise()` | Real sampling variance from a hosted model; same averaging fix |
| Tracing | In-notebook `Tracer` class | `arize-phoenix` (hosted) or `LangSmith` (LangChain's managed tracing SaaS) |

### 1. The Guardrail Still Doesn't Change

Lesson 4's composite guardrail is retrieval-signal-based by
design, precisely so it's independent of whichever embedding/generation stack sits
around it.

### 2. Ragas with a Hosted Judge

## Setting up our RAG prototype

We'll use this to generate answers we can compare to our golden dataset

In [47]:
import os
import subprocess
import time
import requests

OLLAMA_URL = "http://127.0.0.1:11434"


def ollama_is_running() -> bool:
    try:
        response = requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=2,
        )
        return response.ok
    except requests.RequestException:
        return False


if not ollama_is_running():
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
        env={
            **os.environ,
            "OLLAMA_HOST": "127.0.0.1:11434",
        },
    )

    for _ in range(60):
        if ollama_is_running():
            break
        time.sleep(1)
    else:
        raise RuntimeError("Ollama failed to start.")

print("Ollama is running.")

Ollama is running.


In [30]:
!ollama pull llama3.2:1b

In [31]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={
        "normalize_embeddings": True,
    },
)

knowledge_base_texts = [
    document["text"]
    for document in KNOWLEDGE_BASE
]

knowledge_base_metadata = [
    {
        "doc_id": document["doc_id"],
    }
    for document in KNOWLEDGE_BASE
]

dense_store = FAISS.from_texts(
    texts=knowledge_base_texts,
    embedding=embeddings,
    metadatas=knowledge_base_metadata,
)

dense_retriever = dense_store.as_retriever(
    search_kwargs={
        "k": 3,
    }
)

print(f"Indexed {len(KNOWLEDGE_BASE)} documents.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 18 documents.


In [32]:
from langchain_community.llms import Ollama

local_llm = Ollama(
    model="llama3.2:1b",
    temperature=0,
)


def run_local_rag(question: str) -> dict:
    """
    Retrieve context and generate an answer using only the retrieved text.

    Returns the exact fields needed to construct a Ragas dataset.
    """
    retrieved_documents = dense_retriever.invoke(question)

    contexts = [
        document.page_content
        for document in retrieved_documents
    ]

    retrieved_doc_ids = [
        document.metadata.get("doc_id")
        for document in retrieved_documents
    ]

    formatted_context = "\n\n".join(
        f"[Document: {document.metadata.get('doc_id')}]\n"
        f"{document.page_content}"
        for document in retrieved_documents
    )

    prompt = f"""
You are an internal Acme knowledge-base assistant.

Answer the question using only the supplied context.

Rules:
- Do not use outside knowledge.
- Do not invent missing details.
- If the context does not answer the question, say:
  "This information is not available in the knowledge base."
- Keep the answer concise.

Context:
{formatted_context}

Question:
{question}

Answer:
""".strip()

    answer = local_llm.invoke(prompt).strip()

    return {
        "question": question,
        "answer": answer,
        "contexts": contexts,
        "retrieved_doc_ids": retrieved_doc_ids,
    }

In [33]:
example_result = run_local_rag(
    "How do I reset my Acme account password?"
)

print("QUESTION")
print(example_result["question"])

print("\nANSWER")
print(example_result["answer"])

print("\nRETRIEVED DOCUMENT IDS")
print(example_result["retrieved_doc_ids"])

print("\nCONTEXTS")
for context in example_result["contexts"]:
    print("-", context)

QUESTION
How do I reset my Acme account password?

ANSWER
To reset your Acme account password, go to Settings > Security > Reset Password and follow the emailed link.

RETRIEVED DOCUMENT IDS
['kb_001', 'kb_009', 'kb_011']

CONTEXTS
- To reset your Acme account password, go to Settings > Security > Reset Password and follow the emailed link. The link expires after 30 minutes.
- Guests can connect to the 'Acme-Guest' wifi network using the daily password posted at the front desk; it does not require a company account.
- Security incidents, including suspected phishing emails or lost devices, must be reported to security@acme.example within 1 hour of discovery.


In [34]:
questions = []
generated_answers = []
retrieved_contexts = []
reference_answers = []

evaluation_records = []

for example in GOLDEN_DATASET:
    rag_result = run_local_rag(
        example["question"]
    )

    questions.append(
        example["question"]
    )

    generated_answers.append(
        rag_result["answer"]
    )

    retrieved_contexts.append(
        rag_result["contexts"]
    )

    reference_answers.append(
        example["ground_truth"]
    )

    evaluation_records.append({
        "id": example["id"],
        "question": example["question"],
        "answer": rag_result["answer"],
        "contexts": rag_result["contexts"],
        "ground_truth": example["ground_truth"],
        "is_answerable": example["is_answerable"],
        "expected_doc_ids": example["relevant_doc_ids"],
        "retrieved_doc_ids": rag_result["retrieved_doc_ids"],
    })

print(f"Generated answers for {len(evaluation_records)} questions.")

Generated answers for 17 questions.


## Generating Metrics to Evaluate our prototype with

In [62]:
from ragas import evaluate
from ragas.metrics import faithfulness, context_precision, answer_relevancy
from langchain_openai import ChatOpenAI
from datasets import Dataset

hosted_metrics = [
    faithfulness,
    context_precision,
    answer_relevancy,
]

judge = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))

golden_dataset_ragas = Dataset.from_dict({
    "question": questions,
    "answer": generated_answers,
    "contexts": retrieved_contexts,
    "ground_truth": reference_answers,
})

results = evaluate(
    golden_dataset_ragas,
    metrics=hosted_metrics,
    llm=judge)


/tmp/ipykernel_7073/1884605491.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_precision, answer_relevancy
/tmp/ipykernel_7073/1884605491.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, context_precision, answer_relevancy
/tmp/ipykernel_7073/1884605491.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, context_precision, answer_re

Evaluating:   0%|          | 0/51 [00:00<?, ?it/s]

In [61]:
results

{'faithfulness': 0.1569, 'context_precision': 0.7941, 'answer_relevancy': 0.1629}

This is the configuration most Ragas documentation and tutorials assume by default —
a hosted judge is the mainstream real-world setup, which is exactly why the course's
simulated-noise lesson is directly portable here: a real `ChatOpenAI` judge has genuine
run-to-run variance even at `temperature=0`, for the same reasons Lesson 4 discusses
(sampling and infrastructure-level non-determinism in hosted inference).

### 3. Multi-Run Averaging, and Now Rate Limits Too

The course's `stable_evaluate_faithfulness()` averaging pattern transfers directly —
but at cloud scale, running `n_runs=10` over a full evaluation dataset means
`10 × dataset_size` real API calls to the judge model. This is precisely the scenario
Lesson 4's own lesson content flags: **ensure your API key has sufficient funded
quota**, since concurrent judge calls during batch evaluation are a common source of
`429 Rate Limit Exceeded` errors, especially the first time a team runs this at real
dataset scale.

### 4. Cloud Tracing: LangSmith

In [110]:
%pip install --upgrade openai langsmith

  Using cached openai-2.52.0-py3-none-any.whl.metadata (40 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.6/731.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.2/220.2 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 14.2
    Uninstalling websockets-14.2:
      Successfully uninstalled websockets-14.2
  Attempting uninstall: openai
    Found existing installation: openai 1.109.1
    Uninstalling openai-1.109.1:
      Successfully uninstalled openai-1.109.1
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.1.147
    Uninstalling langsmith-0.1.147:
      Successfully uninstalled langsmith-0.1.147
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.2.0 requires webso

In [3]:
import os

# LangSmith: set these env vars for tracing to work
os.environ["LANGSMITH_TRACING"] = "true"
# including LANGSMITH_API_KEY we set above
os.environ["LANGSMITH_PROJECT"] = "rag-course"
os.environ["LANGSMITH_ENDPOINT"]="https://api.smith.langchain.com"


In [4]:
from openai import OpenAI
from langsmith import traceable
from langsmith.wrappers import wrap_openai

client = wrap_openai(OpenAI())

@traceable
def ask_model(question: str):
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": question}],
    )
    return response.choices[0].message.content

print(ask_model("Say hello in five words."))

Hello there, nice to meet you.


**Trade-off:** a hosted tracing backend gives you a shareable dashboard across a whole
team, persistent history beyond a single notebook session, and no local server to keep
running — at the cost of sending trace data (which can include full prompts and
retrieved context) to a third party, which may be a real compliance concern depending
on your corpus.

### What This Buys You, and What It Costs

**Pros:**
- A hosted judge is meaningfully more reliable for nuanced faithfulness/relevancy judgments than a small local model — fewer spot-checks needed before trusting a CI gate built on it
- Hosted tracing gives you team-wide dashboards and persistent history with no infrastructure to maintain
- This is the setup most real production RAG evaluation pipelines actually use, so it's the most directly transferable to a real job

**Cons — all of which the course's lesson content explicitly anticipates:**
- Real, ongoing per-evaluation-run cost, multiplied by however many `n_runs` your averaging strategy uses
- Rate limits are a first-class operational concern for batch evaluation specifically — not a one-off router or embedding call, but potentially thousands of judge calls in a single CI run
- Sending corpus content and generated answers to a third-party judge/tracing service is a real data-governance question worth resolving before adopting this for a regulated or sensitive corpus — precisely the kind of consideration Lesson 4's clinical/financial guardrail examples raise for the guardrail itself, now applying equally to the evaluation and tracing layers

### Bridging Back to the Course

Every lesson in Lesson 4 — the Triad as three separate diagnostic signals, the
necessity of multi-run averaging for any real judge, and span-based tracing turning "it
feels wrong" into an actionable, stage-attributable bug report — holds unchanged. What
a cloud setup adds is real cost and rate-limit exposure at the evaluation layer
specifically, which is exactly the operational risk this module's own lesson content
already warns readers to plan for.